# ECCT on LDPC(49,24) — budget-matched to AECCT

This notebook clones the repository, checks the Kaggle GPU, trains ECCT on the same LDPC
code and model dimensions as AECCT, and prints the resulting BER/FER log.

Enable **Internet** and a **GPU accelerator** in Kaggle before running.

ECCT is trained on **160,000 optimizer updates**, matching the total budget of the
completed AECCT run (400 epochs x 200 batches, spent twice across its FP32 and QAT
phases). Expected wall-clock **~3.3 h**, comfortably inside one Kaggle session — no AECCT
rerun is needed, since this run is brought down to meet the AECCT results already on
disk in `LDPC49_AECCT_results/`.

See `comparison/ECCT_vs_AECCT_LDPC49_COMPARISON.md` for why the previous ECCT run was
not budget-matched and what this run is expected to change.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/gouravanirudh05/SRIP_LDPC_Decoding_using_Machine_Learning.git'
REPO_DIR = Path('/kaggle/working/ldpc_repo')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

ECCT_DIR = REPO_DIR / 'ECCT'
if not (ECCT_DIR / 'Main.py').exists():
    raise FileNotFoundError('ECCT/Main.py is missing from the cloned repository.')

os.chdir(ECCT_DIR)
print('Working directory:', Path.cwd())

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'tqdm'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. In Kaggle, select GPU under Notebook options.')
print('GPU:', torch.cuda.get_device_name(0))

## Training configuration

The code, rate, optimizer, batch size, seed, number of blocks, and embedding dimension
match the completed AECCT run. Only the training budget is set here.

ECCT uses **800 epochs x 200 batches per epoch = 160,000 optimizer updates**, equal to
AECCT's total across its two 400-epoch phases. `Main.py` also evaluates at the halfway
point, so a single run yields both matched comparison points:

| ECCT evaluation | Updates | Matches |
|---|---:|---|
| epoch 400 (mid-run) | 80,000 | AECCT phase-1, FP32 |
| epoch 800 (final) | 160,000 | AECCT final, ternary weights + int8 activations |

`--batch_size` stays at 128. It was already identical in both runs, so changing it would
introduce a new confound rather than remove one — the budget is controlled by `--epochs`
and `--train_batches_per_epoch`.

**`--train_batches_per_epoch=200` must be passed explicitly.** `Main.py` declares
`default=1000`; the earlier ECCT run omitted the flag and silently trained on 500,000
updates, 3.1x AECCT's budget, which invalidated that comparison.

In [ ]:
import time

# Budget-matched to the completed AECCT run (400 epochs x 200 batches PER PHASE,
# two phases = 160,000 optimizer updates total).
#   800 epochs x 200 batches = 160,000 updates  -> matches AECCT total (ternary model)
#   mid-run eval at epoch 400 =  80,000 updates -> matches AECCT phase-1 (FP32 model)
# Estimated wall-clock from the measured 70.34 s / 1k updates: ~3.3 h.
ECCT_EPOCHS = 800

command = [
    sys.executable, 'Main.py',
    '--gpus=0',
    f'--epochs={ECCT_EPOCHS}',
    '--workers=4',
    '--lr=1e-4',
    '--batch_size=128',
    # REQUIRED. Main.py defaults this to 1000; omitting it is what gave the
    # earlier ECCT run 500,000 updates instead of the intended budget.
    '--train_batches_per_epoch=200',
    '--test_batch_size=2048',
    '--seed=42',
    '--code_type=LDPC',
    '--code_n=49',
    '--code_k=24',
    '--N_dec=6',
    '--d_model=128',
    '--h=8',
]
assert '--train_batches_per_epoch=200' in command, 'budget flag missing'
print('Running:', ' '.join(command))
print(f'Optimizer updates: {ECCT_EPOCHS * 200:,} (AECCT total: 160,000)')
start = time.perf_counter()
subprocess.run(command, cwd=str(ECCT_DIR), check=True)
print(f'Total wall time: {(time.perf_counter() - start) / 3600:.2f} hours')

In [ ]:
# Print the latest ECCT result directory and its final log lines.
result_dirs = sorted((ECCT_DIR / 'Results_ECCT').glob('*'), key=lambda p: p.stat().st_mtime)
if not result_dirs:
    raise FileNotFoundError('No ECCT result directory was produced.')
latest = result_dirs[-1]
log_file = latest / 'logging.txt'
print('Result directory:', latest)
print('Checkpoint:', latest / 'best_model')
print('\n'.join(log_file.read_text(errors='replace').splitlines()[-40:]))

In [ ]:
from pathlib import Path
import tarfile

# For ECCT:
result_root = Path("/kaggle/working/ldpc_repo/ECCT/Results_ECCT")

# For AECCT, use instead:
# result_root = Path("/kaggle/working/ldpc_repo/AECCT-main/logs/Results_AECCT")

result_dirs = sorted(result_root.glob("*"), key=lambda p: p.stat().st_mtime)
latest = result_dirs[-1]

archive = Path("/kaggle/working/LDPC49_results.tar.gz")

with tarfile.open(archive, "w:gz") as tar:
    tar.add(latest, arcname=latest.name)

print("Saved:", archive)
print("Log:", latest / "logging.txt")
print("Checkpoint:", latest / "best_model")